# [센서 실습 4] 조도 센서 – BH1750

디지털 조도 센서 **BH1750**의 I2C 레지스터를 직접 읽어 밝기(lux)를 구하고, **히스테리시스**를 적용한 자동 조명 제어를 만들어 봅니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
BH1750 안의 **포토다이오드**는 빛을 받으면 전류가 흐르고, 칩이 이 전류를 16비트 숫자로 바꿉니다.
- 사람 눈과 비슷한 파장 감도를 가지도록 설계되어 결과를 바로 **lux(럭스)** 단위로 쓸 수 있음
- 명령 `0x20`(1회 고해상도 측정)을 보내고 **180ms 기다린 뒤** 2바이트를 읽음
- **lux = (상위바이트 × 256 + 하위바이트) ÷ 1.2**

| 환경 | 밝기 |
|---|---|
| 달빛 | 약 0.1 lux |
| 가로등 아래 | 10 ~ 50 lux |
| 사무실·교실 | 300 ~ 500 lux |
| 흐린 날 실외 | 1,000 ~ 10,000 lux |
| 맑은 날 직사광선 | 30,000 ~ 100,000 lux |

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | 1 ~ 65,535 lux |
| 분해능 | 1 lux (고해상도 모드) |
| 측정 시간 | 약 120~180ms |
| 전원 | 2.4 ~ 3.6V |
| 통신 | I2C 주소 0x23 (ADDR=GND) / 0x5C (ADDR=VCC) |

## 2. 라즈베리파이 연결 방법
**준비물**: BH1750 모듈(GY-302), 점퍼선 4개 (+ 조명 제어 실습 시 LED 또는 릴레이 모듈)

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| BH1750 VCC | 3.3V (1번 핀) |  |
| BH1750 GND | GND (6번 핀) |  |
| BH1750 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| BH1750 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x23) |
| LED(+) – 330Ω 저항 | GPIO18 (12번 핀) | 가로등 역할 (선택) |



**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install smbus2 gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
센서에서 받은 **2바이트**를 lux로 바꾸는 식입니다.

In [ ]:
def bh1750_bytes_to_lux(data):
    """BH1750이 보낸 [상위바이트, 하위바이트] → lux"""
    return ((data[0] << 8) | data[1]) / 1.2

print(bh1750_bytes_to_lux([0x01, 0xE0]), "lux")   # 0x01E0 = 480 → 400 lux

### 4-2. 실제 센서 클래스
`smbus2`로 측정 명령을 보낸 뒤 2바이트를 읽습니다. 라이브러리 없이 **레지스터 수준에서 통신하는 방법**을 볼 수 있습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealBH1750:
    ONE_TIME_HIGH_RES = 0x20          # 1회 고해상도 측정 명령

    def __init__(self, address=0x23, bus=1):
        from smbus2 import SMBus
        self.bus, self.addr = SMBus(bus), address

    def read(self):
        from smbus2 import i2c_msg
        self.bus.write_byte(self.addr, self.ONE_TIME_HIGH_RES)
        time.sleep(0.18)                                 # 측정 완료까지 대기
        msg = i2c_msg.read(self.addr, 2)
        self.bus.i2c_rdwr(msg)
        data = list(msg)
        return {"raw": (data[0] << 8) | data[1], "lux": round(bh1750_bytes_to_lux(data), 1)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 장소 | 창가에 설치한 센서 | 실내 + 바깥 빛 영향 |
| 일출·일몰 | 6시 · 18시 | 봄·가을 기준 |
| 한낮 최대 | 약 1,200 lux | 창가 실내 밝기 |
| 구름 | 무작위로 30~60% 어두워짐 | 날씨 변화 |
| 밤 | 0~3 lux | 주변 불빛 |

가상 센서는 lux를 **실제 센서처럼 2바이트(raw)로 바꿔서** 변환식에 넘깁니다.

In [ ]:
class VirtualBH1750:
    def __init__(self, dt=600, seed=4):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)
        self.cloud = 1.0

    def read(self):
        hour = (self.t / 3600) % 24
        sun = max(0.0, np.sin(np.pi * (hour - 6) / 12))           # 6시~18시 해
        if self.rng.random() < 0.1:                               # 구름 상태가 가끔 바뀜
            self.cloud = self.rng.choice([1.0, 0.7, 0.4])
        lux = 1200 * sun * self.cloud + self.rng.uniform(0, 3)
        raw = int(np.clip(lux * 1.2, 0, 65535))
        data = [raw >> 8, raw & 0xFF]                             # 실제 센서와 같은 2바이트
        self.t += self.dt
        return {"raw": raw, "lux": round(bh1750_bytes_to_lux(data), 1)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualBH1750(dt=600)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
10분 간격으로 24시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 144               # 측정 횟수
INTERVAL = 600               # 측정 간격(초)

sensor = RealBH1750() if USE_REAL_SENSOR else VirtualBH1750(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
**히스테리시스(hysteresis)** 를 적용한 가로등 제어입니다.
- 50 lux **아래로** 떨어지면 켜고, 80 lux **위로** 올라가야 끔
- 기준이 하나뿐이면 50 lux 근처에서 구름이 지날 때마다 조명이 깜빡거리므로, 켜는 기준과 끄는 기준을 다르게 둡니다.

In [ ]:
df["hour"] = df["time_s"] / 3600
ON_LUX, OFF_LUX = 50, 80
light, states = False, []
for lux in df["lux"]:
    if not light and lux < ON_LUX:
        light = True
    elif light and lux > OFF_LUX:
        light = False
    states.append(int(light))
df["light_on"] = states

print(f"{df['hour'].iloc[0]:5.1f}시  조명 {'켜짐' if df['light_on'].iloc[0] else '꺼짐'}  (측정 시작)")
changes = df["light_on"].diff().fillna(0)
for _, r in df[changes != 0].iterrows():
    print(f"{r['hour']:5.1f}시  조명 {'켜짐' if r['light_on'] else '꺼짐'}  ({r['lux']:.0f} lux)")
print(f"조명 켜진 시간: {df['light_on'].sum() * INTERVAL / 3600:.1f}시간")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(df["hour"], df["lux"], "k-", lw=1.5, label="조도")
ax[0].axhline(ON_LUX, color="gray", ls="--", label="켜짐 기준 50 lux")
ax[0].axhline(OFF_LUX, color="gray", ls=":", label="꺼짐 기준 80 lux")
ax[0].set_ylabel("lux"); ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].set_title("BH1750 조도와 자동 조명 제어")
ax[1].step(df["hour"], df["light_on"], "k-", where="post")
ax[1].fill_between(df["hour"], df["light_on"], step="post", color="lightgray")
ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["꺼짐", "켜짐"]); ax[1].set_xlabel("시각 (시)")
plt.xticks(range(0, 25, 3)); plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "광조도센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "BH1750",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/light", payload)
# client.disconnect()

## 직접 해 보기
- `OFF_LUX`를 50으로 바꿔 히스테리시스를 없애면 조명 전환 횟수가 어떻게 되나요?
- 교실 권장 밝기(400 lux)에 못 미치는 시간대를 찾아 '보조 조명 필요'로 표시해 보세요.
- 명령을 `0x21`(고해상도 모드2, 0.5 lux 분해능)로 바꾸면 변환식은 어떻게 달라질까요? (÷1.2 후 ÷2)